In [ ]:
!pip install -q pyroomacoustics soundfile scipy


In [ ]:
import numpy as np
import pyroomacoustics as pra
import soundfile as sf
from IPython.display import Audio, display


In [ ]:
# Room geometry (meters): [Length, Width, Height]
room_dim = [4.5, 3.6, 2.8]

# Simulation settings
fs = 16000
max_order = 20

np.random.seed(42)

In [ ]:
# BEFORE: more reflective
alpha_before = 0.02

# AFTER: more absorptive
alpha_after = 0.55


In [ ]:
# Source and microphone positions [x, y, z] in meters
source_pos = [1.0, 1.0, 1.5]
mic_pos    = [3.5, 2.5, 1.5]


In [ ]:
def rt60_schroeder(rir, fs, t_start_db=-5, t_end_db=-35):

    rir = np.array(rir, dtype=float)
    if rir.size == 0:
        return None

    # Normalise
    rir = rir / (np.max(np.abs(rir)) + 1e-12)

    # Energy decay curve (Schroeder integration)
    edc = np.cumsum(rir[::-1]**2)[::-1]
    edc = edc / (np.max(edc) + 1e-12)
    edc_db = 10 * np.log10(edc + 1e-12)

    # Find fitting region
    idx_start = np.where(edc_db <= t_start_db)[0]
    idx_end = np.where(edc_db <= t_end_db)[0]
    if len(idx_start) == 0 or len(idx_end) == 0:
        return None

    i1 = idx_start[0]
    i2 = idx_end[0]
    if i2 <= i1 + 10:
        return None

    t = np.arange(len(rir)) / fs
    x = t[i1:i2]
    y = edc_db[i1:i2]

    # Linear regression y = a*x + b
    a, b = np.polyfit(x, y, 1)
    if a >= 0:
        return None

    # RT60 = time for -60 dB decay
    return float(-60.0 / a)


In [ ]:
def safe_normalize(x, peak=0.95):
    x = np.array(x, dtype=float)
    m = np.max(np.abs(x)) + 1e-12
    return (x / m) * peak


In [ ]:
room_before = pra.ShoeBox(
    room_dim,
    fs=fs,
    materials=pra.Material(float(alpha_before)),
    max_order=max_order
)

room_before.add_source(source_pos)
room_before.add_microphone_array(
    pra.MicrophoneArray(np.array([mic_pos]).T, fs)
)

room_before.compute_rir()
rir_before = room_before.rir[0][0]

rt60_before = rt60_schroeder(rir_before, fs)
print("RT60 BEFORE (Schroeder):", rt60_before)


RT60 BEFORE (Schroeder): 0.2793504861090525


In [ ]:
room_after = pra.ShoeBox(
    room_dim,
    fs=fs,
    materials=pra.Material(float(alpha_after)),
    max_order=max_order
)

room_after.add_source(source_pos)
room_after.add_microphone_array(
    pra.MicrophoneArray(np.array([mic_pos]).T, fs)
)

room_after.compute_rir()
rir_after = room_after.rir[0][0]

rt60_after = rt60_schroeder(rir_after, fs)
print("RT60 AFTER (Schroeder):", rt60_after)


RT60 AFTER (Schroeder): 0.14515743935971992


In [ ]:
# Click/impulse signal
dry_click = np.zeros(int(fs * 2.0))
# dry_click[0] = 1.0
dry_click[:20] = np.hanning(40)[:20]  # tiny smoothed "click"

before_click = np.convolve(dry_click, rir_before)
after_click  = np.convolve(dry_click, rir_after)

before_click = safe_normalize(before_click, 0.95)
after_click  = safe_normalize(after_click, 0.95)

sf.write("before_click.wav", before_click, fs)
sf.write("after_click.wav", after_click, fs)

print("Saved: before_click.wav, after_click.wav")


Saved: before_click.wav, after_click.wav


In [ ]:
# Create a voice-ish dry signal (band-limited noise: 120 Hz – 3500 Hz)
duration = 3.0
N = int(fs * duration)

white = np.random.randn(N)

X = np.fft.rfft(white)
freqs = np.fft.rfftfreq(N, d=1/fs)
mask = (freqs >= 120) & (freqs <= 3500)
X[~mask] = 0
dry_voiceish = np.fft.irfft(X, n=N)

dry_voiceish = safe_normalize(dry_voiceish, 0.30)

before_real = np.convolve(dry_voiceish, rir_before)
after_real  = np.convolve(dry_voiceish, rir_after)

before_real = safe_normalize(before_real, 0.95)
after_real  = safe_normalize(after_real, 0.95)

sf.write("before_real.wav", before_real, fs)
sf.write("after_real.wav", after_real, fs)

print("Saved: before_real.wav, after_real.wav")


Saved: before_real.wav, after_real.wav


In [ ]:
print("BEFORE (click response)")
display(Audio("before_click.wav"))

print("AFTER (click response)")
display(Audio("after_click.wav"))

print("BEFORE (realistic auralization)")
display(Audio("before_real.wav"))

print("AFTER (realistic auralization)")
display(Audio("after_real.wav"))

print(f"RT60 reduction: {rt60_before - rt60_after:.3f} s ({(1-rt60_after/rt60_before)*100:.1f}%)")



BEFORE (click response)


AFTER (click response)


BEFORE (realistic auralization)


AFTER (realistic auralization)


RT60 reduction: 0.134 s (48.0%)


In [ ]:
import soundfile as sf
import numpy as np
from IPython.display import Audio, display
import scipy.signal as sps
import os

if not os.path.exists("dry.wav"):
    print("dry.wav not found. Upload a clean dry WAV file (Files panel → Upload) and re-run this cell.")
else:
    dry, sr = sf.read("dry.wav")

# Convert to mono if stereo
if dry.ndim > 1:
    dry = dry.mean(axis=1)

# Resample if needed
if sr != fs:
    dry = sps.resample_poly(dry, fs, sr)

# Trim to first 6 seconds
dry = dry[:fs * 6]

# Convolve with room impulse responses
before_real = np.convolve(dry, rir_before)
after_real  = np.convolve(dry, rir_after)

# Safe normalization
before_real = safe_normalize(before_real, 0.95)
after_real  = safe_normalize(after_real, 0.95)

# Save files
sf.write("before_real.wav", before_real, fs)
sf.write("after_real.wav", after_real, fs)

print("Saved: before_real.wav and after_real.wav")

print("BEFORE (auralized)")
display(Audio("before_real.wav"))

print("AFTER (auralized)")
display(Audio("after_real.wav"))


Saved: before_real.wav and after_real.wav
BEFORE (auralized)


AFTER (auralized)
